# MO-IT134 Milestone 2: Project Forecasting Model with DataViz (Improved Final Revision)

## Objective
This notebook revises the Milestone 2 draft based on feedback. The goal is to build a clearer, fully documented, end-to-end forecasting workflow for **FinMark monthly revenue** and produce a **6-month forecast**.

## Milestone 2 Improved
- Added detailed documentation from start to finish
- Cleaned the workflow so it runs end-to-end in Google Colab
- Improved the six-month forecasting section
- Refined and labeled visualizations
- Evaluated **another algorithm** and compared results
- Corrected technical issues from the original draft and Section 12 Random Forest file

## Models compared
1. **SARIMAX** — time series forecasting model  
2. **Random Forest Regressor** — adapted from the Section 12 idea, but revised so it predicts the **same target** as SARIMAX: monthly revenue

## Final output
- Model comparison table
- Backtest results for the last 6 months
- Final 6-month revenue forecast
- CSV exports for forecast and monthly history

## Step 1 — Upload the dataset zip file

 Run and upload:

`-Machine-Learning-and-Predictive-Analytics-main.zip`



In [ ]:
from google.colab import files
uploaded = files.upload()

## Step 2 — Import libraries and unzip the project files

In [ ]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

zip_path = "-Machine-Learning-and-Predictive-Analytics-main.zip"
extract_path = "finmark_project"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully.")
print(os.listdir(extract_path))

## Step 3 — Load the preprocessed merged dataset

The `merged_data.csv` file combines transaction, company, and product information.  
This lets us compute a revenue target and then aggregate the data monthly for forecasting.

In [ ]:
file_path = "finmark_project/-Machine-Learning-and-Predictive-Analytics-main/merged_data.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

## Step 4 — Re-check the dataset structure

This section satisfies the requirement to examine the dataset, re-check the features, and verify their data types before modelling.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

## Step 5 — Data cleaning and target creation

We convert the transaction date to datetime, convert numeric columns properly, and compute the forecasting target:

**Revenue = Quantity × Unit Price**

This is used as the main target for the monthly forecasting model.

In [ ]:
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
df["Unit_Price"] = pd.to_numeric(df["Product_Price_x"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Transaction_ID"] = pd.to_numeric(df["Transaction_ID"], errors="coerce")
df["Company_ID"] = pd.to_numeric(df["Company_ID"], errors="coerce")
df["Product_ID"] = pd.to_numeric(df["Product_ID"], errors="coerce")
df["Company_Profit"] = pd.to_numeric(df["Company_Profit"], errors="coerce")

df["Revenue"] = df["Quantity"] * df["Unit_Price"]

df = df.dropna(subset=["Transaction_Date", "Quantity", "Unit_Price", "Revenue"]).copy()
df = df[df["Revenue"] >= 0].copy()

print("Cleaned shape:", df.shape)
df[["Transaction_Date", "Quantity", "Unit_Price", "Revenue"]].head()

## Step 6 — Feature engineering

We add calendar-based features and rolling indicators to strengthen the analysis and improve interpretability.

In [ ]:
df["Year"] = df["Transaction_Date"].dt.year
df["Month"] = df["Transaction_Date"].dt.month
df["Quarter"] = df["Transaction_Date"].dt.quarter
df["DayOfWeek"] = df["Transaction_Date"].dt.dayofweek
df["Is_Weekend"] = (df["DayOfWeek"] >= 5).astype(int)

df = df.sort_values(["Company_ID", "Transaction_Date", "Transaction_ID"]).reset_index(drop=True)

company_roll = (
    df.groupby("Company_ID")
      .rolling("30D", on="Transaction_Date")["Revenue"]
      .sum()
      .reset_index(level=0, drop=True)
)
df["Company_30D_Revenue_Rolling"] = company_roll.to_numpy()

df = df.sort_values(["Product_ID", "Transaction_Date", "Transaction_ID"]).reset_index(drop=True)

product_roll = (
    df.groupby("Product_ID")
      .rolling("30D", on="Transaction_Date")["Quantity"]
      .sum()
      .reset_index(level=0, drop=True)
)
df["Product_30D_Quantity_Rolling"] = product_roll.to_numpy()

df[[
    "Transaction_Date",
    "Company_ID",
    "Product_ID",
    "Revenue",
    "Company_30D_Revenue_Rolling",
    "Product_30D_Quantity_Rolling"
]].head()

## Step 7 — Aggregate the data by month

The forecasting target is **monthly revenue**, not daily or transaction-level revenue.  
We also keep related monthly signals such as:
- total quantity
- number of transactions
- unique companies
- unique products

In [ ]:
monthly = (
    df.set_index("Transaction_Date")
      .resample("MS")
      .agg({
          "Revenue": "sum",
          "Quantity": "sum",
          "Transaction_ID": "count",
          "Company_ID": pd.Series.nunique,
          "Product_ID": pd.Series.nunique
      })
      .rename(columns={
          "Transaction_ID": "Transactions",
          "Company_ID": "Unique_Companies",
          "Product_ID": "Unique_Products"
      })
)

monthly["MonthNum"] = monthly.index.month
monthly["Quarter"] = monthly.index.quarter
monthly["Year"] = monthly.index.year

print("Monthly shape:", monthly.shape)
monthly.head()

## Step 8 — Exploratory visualizations

These visualizations help explain the historical revenue trend, relationships among variables, and seasonality patterns.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly.index, monthly["Revenue"], marker="o")
plt.title("FinMark Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(monthly[["Revenue", "Quantity", "Transactions", "Unique_Companies", "Unique_Products"]].corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation of Monthly Revenue Variables")
plt.show()

In [ ]:
seasonality = monthly.groupby("MonthNum")["Revenue"].mean()

plt.figure(figsize=(8, 4))
seasonality.plot(kind="bar")
plt.title("Average Revenue by Month of Year")
plt.xlabel("Month Number")
plt.ylabel("Average Revenue")
plt.xticks(rotation=0)
plt.show()

## Step 9 — Set up the 6-month backtest

To evaluate forecasting performance, we reserve the **last 6 months** as the test period and train the models on the earlier months.

In [ ]:
test_horizon = 6

train_monthly = monthly.iloc[:-test_horizon].copy()
test_monthly = monthly.iloc[-test_horizon:].copy()

print("Training months:", len(train_monthly))
print("Testing months:", len(test_monthly))
print("\nTest period:")
display(test_monthly[["Revenue", "Quantity", "Transactions"]])

## Step 10 — Model 1: SARIMAX

SARIMAX is used as the primary time series model.  
A log transform is applied to stabilize the scale of monthly revenue.

In [ ]:
train_y_log = np.log1p(train_monthly["Revenue"])

sarimax_model = SARIMAX(
    train_y_log,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarimax_forecast_obj = sarimax_model.get_forecast(steps=test_horizon)
sarimax_pred = np.expm1(sarimax_forecast_obj.predicted_mean)
sarimax_ci = np.expm1(sarimax_forecast_obj.conf_int())

sarimax_pred.index = test_monthly.index
sarimax_ci.index = test_monthly.index

sarimax_results = pd.DataFrame({
    "Actual_Revenue": test_monthly["Revenue"],
    "SARIMAX_Predicted": sarimax_pred
})

sarimax_results

## Step 11 — Model 2: Random Forest (adapted from Section 12)

Here, the Random Forest approach is it predicts the **same monthly revenue target** as SARIMAX.

To make the Random Forest useful for forecasting, we create lag and rolling features from past monthly values.

In [ ]:
rf_data = monthly.copy()

for lag in [1, 2, 3, 6, 12]:
    rf_data[f"Revenue_Lag_{lag}"] = rf_data["Revenue"].shift(lag)
    rf_data[f"Quantity_Lag_{lag}"] = rf_data["Quantity"].shift(lag)
    rf_data[f"Transactions_Lag_{lag}"] = rf_data["Transactions"].shift(lag)

rf_data["Revenue_Rolling_3"] = rf_data["Revenue"].shift(1).rolling(3).mean()
rf_data["Revenue_Rolling_6"] = rf_data["Revenue"].shift(1).rolling(6).mean()
rf_data["Quantity_Rolling_3"] = rf_data["Quantity"].shift(1).rolling(3).mean()
rf_data["Transactions_Rolling_3"] = rf_data["Transactions"].shift(1).rolling(3).mean()

rf_data = rf_data.dropna().copy()

rf_train = rf_data.iloc[:-test_horizon].copy()
rf_test = rf_data.iloc[-test_horizon:].copy()

feature_cols = [col for col in rf_data.columns if col != "Revenue"]

X_train = rf_train[feature_cols]
y_train = rf_train["Revenue"]
X_test = rf_test[feature_cols]
y_test = rf_test["Revenue"]

rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = pd.Series(rf_model.predict(X_test), index=rf_test.index, name="RF_Predicted")

rf_results = pd.DataFrame({
    "Actual_Revenue": y_test,
    "RF_Predicted": rf_pred
})

rf_results

## Step 12 — Compare model performance

We evaluate both models using:
- **MAE**: Mean Absolute Error
- **RMSE**: Root Mean Squared Error
- **MAPE**: Mean Absolute Percentage Error

This directly addresses the feedback asking for evaluation of other algorithms and comparison of results.

In [ ]:
def compute_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / np.maximum(actual, 1e-9))) * 100
    return mae, rmse, mape

sarimax_mae, sarimax_rmse, sarimax_mape = compute_metrics(
    test_monthly["Revenue"], sarimax_pred
)

rf_mae, rf_rmse, rf_mape = compute_metrics(
    y_test, rf_pred
)

comparison = pd.DataFrame({
    "Model": ["SARIMAX", "Random Forest"],
    "MAE": [sarimax_mae, rf_mae],
    "RMSE": [sarimax_rmse, rf_rmse],
    "MAPE (%)": [sarimax_mape, rf_mape]
}).sort_values("MAPE (%)")

comparison

## Step 13 — Backtest visualization

This chart helps show whether the forecast pattern is reasonable compared with the actual values from the final 6 months.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_monthly.index, train_monthly["Revenue"], label="Training Revenue")
plt.plot(test_monthly.index, test_monthly["Revenue"], label="Actual Revenue", marker="o")
plt.plot(sarimax_pred.index, sarimax_pred, label="SARIMAX Prediction", marker="o")
plt.plot(rf_pred.index, rf_pred, label="Random Forest Prediction", marker="o")

plt.title("Backtest Comparison: Actual vs Predicted Revenue (Last 6 Months)")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 14 — Residual check for SARIMAX

This step checks whether the time series model still has unexplained structure in the residuals.

In [ ]:
resid = sarimax_model.resid

plt.figure(figsize=(12, 4))
pd.Series(resid, index=train_monthly.index).plot()
plt.title("SARIMAX Residuals (Training Period)")
plt.xlabel("Month")
plt.ylabel("Residual (log scale)")
plt.grid(True, alpha=0.3)
plt.show()

acorr_ljungbox(resid, lags=[6, 12], return_df=True)

## Step 15 — Select the better model

The model with the lower MAPE is selected for the final 6-month forecast.

In [ ]:
best_model_name = comparison.iloc[0]["Model"]
print("Best model based on lowest MAPE:", best_model_name)
comparison

## Step 16 — Build the final 6-month forecast

We now retrain the selected model using the **full monthly history** and generate a forecast for the next 6 months.

In [ ]:
if best_model_name == "SARIMAX":
    final_model = SARIMAX(
        np.log1p(monthly["Revenue"]),
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    final_forecast_obj = final_model.get_forecast(steps=6)
    final_mean = np.expm1(final_forecast_obj.predicted_mean)
    final_ci = np.expm1(final_forecast_obj.conf_int())

    forecast_index = pd.date_range(
        start=monthly.index.max() + pd.offsets.MonthBegin(1),
        periods=6,
        freq="MS"
    )

    forecast_df = pd.DataFrame({
        "Forecast_Revenue": final_mean.values,
        "CI_Lower_95": final_ci.iloc[:, 0].values,
        "CI_Upper_95": final_ci.iloc[:, 1].values
    }, index=forecast_index)

else:
    final_rf = rf_data.copy()

    # Rebuild lag features already done above in rf_data and fit on full available data
    feature_cols = [col for col in final_rf.columns if col != "Revenue"]

    final_rf_model = RandomForestRegressor(
        n_estimators=400,
        max_depth=8,
        random_state=42
    )
    final_rf_model.fit(final_rf[feature_cols], final_rf["Revenue"])

    # Recursive forecast for 6 future months
    history = monthly.copy()
    future_rows = []

    for _ in range(6):
        next_idx = history.index.max() + pd.offsets.MonthBegin(1)

        new_row = {
            "Quantity": history["Quantity"].iloc[-1],
            "Transactions": history["Transactions"].iloc[-1],
            "Unique_Companies": history["Unique_Companies"].iloc[-1],
            "Unique_Products": history["Unique_Products"].iloc[-1],
            "MonthNum": next_idx.month,
            "Quarter": next_idx.quarter,
            "Year": next_idx.year,
        }

        temp = pd.concat([history, pd.DataFrame([new_row], index=[next_idx])], axis=0)

        for lag in [1, 2, 3, 6, 12]:
            temp.loc[next_idx, f"Revenue_Lag_{lag}"] = temp["Revenue"].shift(lag).loc[next_idx]
            temp.loc[next_idx, f"Quantity_Lag_{lag}"] = temp["Quantity"].shift(lag).loc[next_idx]
            temp.loc[next_idx, f"Transactions_Lag_{lag}"] = temp["Transactions"].shift(lag).loc[next_idx]

        temp.loc[next_idx, "Revenue_Rolling_3"] = temp["Revenue"].shift(1).rolling(3).mean().loc[next_idx]
        temp.loc[next_idx, "Revenue_Rolling_6"] = temp["Revenue"].shift(1).rolling(6).mean().loc[next_idx]
        temp.loc[next_idx, "Quantity_Rolling_3"] = temp["Quantity"].shift(1).rolling(3).mean().loc[next_idx]
        temp.loc[next_idx, "Transactions_Rolling_3"] = temp["Transactions"].shift(1).rolling(3).mean().loc[next_idx]

        row_features = temp.loc[[next_idx], feature_cols]
        predicted_revenue = final_rf_model.predict(row_features)[0]

        temp.loc[next_idx, "Revenue"] = predicted_revenue
        history = temp.copy()

        future_rows.append((next_idx, predicted_revenue))

    forecast_df = pd.DataFrame(
        {"Forecast_Revenue": [v for _, v in future_rows]},
        index=[k for k, _ in future_rows]
    )

forecast_df

## Step 17 — Final stakeholder visualizations

These charts present the historical revenue together with the projected revenue for the next 6 months.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly.index, monthly["Revenue"], label="Historical Revenue")
plt.plot(forecast_df.index, forecast_df["Forecast_Revenue"], label="6-Month Forecast", marker="o")

if "CI_Lower_95" in forecast_df.columns and "CI_Upper_95" in forecast_df.columns:
    plt.fill_between(
        forecast_df.index,
        forecast_df["CI_Lower_95"],
        forecast_df["CI_Upper_95"],
        alpha=0.2,
        label="95% Confidence Interval"
    )

plt.title("FinMark Monthly Revenue — Final 6-Month Forecast")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
forecast_df["Forecast_Revenue"].plot(kind="bar")
plt.title("Projected Monthly Revenue for the Next 6 Months")
plt.xlabel("Forecast Month")
plt.ylabel("Forecast Revenue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 18 — Save final outputs

The notebook saves the following files:
- `finmark_6_month_forecast_final.csv`
- `finmark_monthly_history_final.csv`
- `model_comparison_results.csv`

In [ ]:
forecast_df.to_csv("finmark_6_month_forecast_final.csv")
monthly.to_csv("finmark_monthly_history_final.csv")
comparison.to_csv("model_comparison_results.csv", index=False)

print("Saved:")
print("- finmark_6_month_forecast_final.csv")
print("- finmark_monthly_history_final.csv")
print("- model_comparison_results.csv")

## Conclusion

This revised and improved notebook addresses the feedback by improving the clarity, documentation, modelling consistency, and technical reliability of the Milestone 2 submission.

The workflow now:
- runs from start to finish in Google Colab
- includes detailed explanations
- compares two algorithms on the **same forecasting target**
- evaluates the models using the last 6 months as a backtest period
- produces a final 6-month forecast with clear visualizations

